# Unified Training Example

This notebook demonstrates how to use the updated `train_model_unified` function to train both feedforward and sequence models.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().cwd().parent
sys.path.insert(0, str(project_root))

from src.model_utils import train_model_unified
from src.model import MODEL_TYPE

## Configuration

In [ ]:
# Common configuration
store_cluster = 17
item_cluster = 15
label_cols = ["unit_sales"]
y_to_log_features = ["unit_sales"]
epochs = 30
lr = 3e-4
log_level = "INFO"

# Directories
model_dir = Path("../output/models/unified/")
model_logger_dir = Path("../output/logs/unified/")
history_dir = Path("../output/history/unified/")

print(f"Training models for store_cluster={store_cluster}, item_cluster={item_cluster}")
print(f"Epochs: {epochs}, Learning rate: {lr}")

## 1. Train Feedforward Models

In [ ]:
# Feedforward model configuration
feedforward_dataloader_dir = Path("../output/data/dataloader_2014_2015_top_53_store_2000_item/")

# Train ShallowNN
print("\n" + "="*50)
print("Training ShallowNN")
print("="*50)

train_model_unified(
    model_dir=model_dir,
    dataloader_dir=feedforward_dataloader_dir,
    model_logger_dir=model_logger_dir,
    model_type=ModelType.SHALLOW_NN,  # ModelType enum for feedforward
    model_family="feedforward",
    label_cols=label_cols,
    y_to_log_features=y_to_log_features,
    store_cluster=store_cluster,
    item_cluster=item_cluster,
    history_dir=history_dir,
    lr=lr,
    epochs=epochs,
    hidden_dim=128,
    dropout=0.2,
    log_level=log_level,
)

In [ ]:
# Train TwoLayerNN
print("\n" + "="*50)
print("Training TwoLayerNN")
print("="*50)

train_model_unified(
    model_dir=model_dir,
    dataloader_dir=feedforward_dataloader_dir,
    model_logger_dir=model_logger_dir,
    model_type=ModelType.TWO_LAYER_NN,  # ModelType enum for feedforward
    model_family="feedforward",
    label_cols=label_cols,
    y_to_log_features=y_to_log_features,
    store_cluster=store_cluster,
    item_cluster=item_cluster,
    history_dir=history_dir,
    lr=lr,
    epochs=epochs,
    h1=64,
    h2=32,
    dropout=0.4,
    log_level=log_level,
)

## 2. Train Sequence Models

In [ ]:
# Sequence model configuration
sequence_dataloader_dir = Path("../output/data/dataloader_seq_model_2014_2015_top_53_store_2000_item/")
sequence_lr = 1e-3  # Different LR for sequence models

# Train TFT (Temporal Fusion Transformer)
print("\n" + "="*50)
print("Training TFT (Temporal Fusion Transformer)")
print("="*50)

train_model_unified(
    model_dir=model_dir,
    dataloader_dir=sequence_dataloader_dir,
    model_logger_dir=model_logger_dir,
    model_type=SEQUENCE_MODEL.TFT,  # String for sequence models
    model_family="sequence",
    label_cols=label_cols,
    y_to_log_features=y_to_log_features,
    store_cluster=store_cluster,
    item_cluster=item_cluster,
    history_dir=history_dir,
    lr=sequence_lr,
    epochs=epochs,
    hidden_dim=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    log_level=log_level,
)

In [ ]:
# Train NBEATS
print("\n" + "="*50)
print("Training NBEATS")
print("="*50)

train_model_unified(
    model_dir=model_dir,
    dataloader_dir=sequence_dataloader_dir,
    model_logger_dir=model_logger_dir,
    model_type=SEQUENCE_MODEL.NBEATS,  # String for sequence models
    model_family="sequence",
    label_cols=label_cols,
    y_to_log_features=y_to_log_features,
    store_cluster=store_cluster,
    item_cluster=item_cluster,
    history_dir=history_dir,
    lr=sequence_lr,
    epochs=epochs,
    hidden_dim=32,
    dropout=0.1,
    log_level=log_level,
)

In [ ]:
# Train DeepAR
print("\n" + "="*50)
print("Training DeepAR")
print("="*50)

train_model_unified(
    model_dir=model_dir,
    dataloader_dir=sequence_dataloader_dir,
    model_logger_dir=model_logger_dir,
    model_type=SEQUENCE_MODEL.DEEPAR,  # String for sequence models
    model_family="sequence",
    label_cols=label_cols,
    y_to_log_features=y_to_log_features,
    store_cluster=store_cluster,
    item_cluster=item_cluster,
    history_dir=history_dir,
    lr=sequence_lr,
    epochs=epochs,
    hidden_dim=32,
    dropout=0.1,
    log_level=log_level,
)

## 3. Compare Results

In [ ]:
import pandas as pd
import glob

# Load all history files
history_files = glob.glob(str(history_dir / f"{store_cluster}_{item_cluster}_*.csv"))

if history_files:
    all_history = []
    for file in history_files:
        df = pd.read_csv(file)
        all_history.append(df)
    
    combined_history = pd.concat(all_history, ignore_index=True)
    
    print("\n" + "="*50)
    print("TRAINING RESULTS COMPARISON")
    print("="*50)
    
    # Sort by validation loss
    if 'best_val_loss' in combined_history.columns:
        combined_history = combined_history.sort_values('best_val_loss')
        print(combined_history[['model_name', 'best_val_loss']].to_string(index=False))
    else:
        print(combined_history.to_string(index=False))
else:
    print("No history files found.")

## Key Differences Between Model Families

### Feedforward Models
- **Input**: `model_type` as `ModelType` enum (e.g., `ModelType.SHALLOW_NN`)
- **Data**: Uses regular dataloaders with tabular features
- **Parameters**: `hidden_dim`, `h1`, `h2`, `depth`, `dropout`

### Sequence Models  
- **Input**: `model_type` as string (e.g., `SEQUENCE_MODEL.TFT`)
- **Data**: Uses sequence dataloaders with time series structure
- **Parameters**: `hidden_dim`, `attention_head_size`, `hidden_continuous_size`, `dropout`

### Available Sequence Models
- **TFT**: Temporal Fusion Transformer - best for complex temporal patterns
- **NBEATS**: Neural Basis Expansion Analysis - good for univariate forecasting
- **DeepAR**: Probabilistic forecasting with RNN
- **LSTM**: Simple LSTM-based forecasting